# 03 · Anotace, leakage a kalibrace

Tyto notebooky používají skutečné soubory projektu, ne vymyšlené výsledky. Buňky bez dat skončí vysvětlením, které balíčky chybí. Klon repozitáře může vyžadovat přístup; alternativně nahraj jeho kopii do Colab. API klíč pro tuto analýzu nepotřebuješ. Trénování ani nové vyhledávání se bez přepínače nespouští. Rezervované testové otázky se nečtou ani nevyhodnocují.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
# In Colab clone the project; locally reuse the current checkout.
if Path('src/librarian').is_dir():
    ROOT = Path.cwd()
elif Path('ML_Librarian/src/librarian').is_dir():
    ROOT = Path('ML_Librarian').resolve()
else:
    subprocess.run(['git','clone','https://github.com/Kilitar/ML_Librarian.git'],check=True)
    ROOT = Path('ML_Librarian').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src/librarian'))
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
# Set an immutable git ref before starting for exact code-version reproduction.


In [ ]:
# Optional environment setup. Enable in a fresh Colab runtime; CPU is sufficient.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-librarian.txt'],check=True)


In [ ]:
# Operator supplies the ignored data bundle(s), never an API key.
UPLOAD_BUNDLES = False  # In Colab set True and select runtime.local.zip and/or development.local.zip.
if UPLOAD_BUNDLES:
    from google.colab import files
    uploaded = files.upload()
    import io, zipfile
    expected = json.loads((ROOT/'deployment/artifacts.json').read_text())
    for filename, blob in uploaded.items():
        record = expected.get(filename)
        if record is None or hashlib.sha256(blob).hexdigest() != record['sha256']:
            raise ValueError('Unknown/stale bundle: compare with operator manifest; do not bypass SHA check')
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            names=z.namelist()
            manifest=json.loads(z.read('bundle_manifest.json'))
            if len(names)!=len(set(names)) or set(names)!=set(manifest)|{'bundle_manifest.json'}:
                raise ValueError('Invalid bundle manifest')
            if sum(i.file_size for i in z.infolist())>40*1024*1024:raise ValueError('Bundle too large')
            staged=[]
            for name, digest in manifest.items():
                if name not in record['files']:raise ValueError('Unexpected member')
                target=(ROOT/name).resolve()
                if not target.is_relative_to(ROOT.resolve()):raise ValueError('Unsafe path')
                content=z.read(name)
                if hashlib.sha256(content).hexdigest()!=digest:raise ValueError('Member SHA mismatch')
                staged.append((target,content))
            for target,content in staged:
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
        print('Verified bundle:',filename)


In [ ]:
from collections import Counter
from calibration_data import group_pairs, coverage
path=ROOT/'eval/source_group_extension.local.json'
if not path.exists():
    print('Chybí development.local.zip: skutečné páry nelze nahradit syntetickými metrikami.')
else:
    rows=json.loads(path.read_text(encoding='utf-8'))['rows']
    grouped=group_pairs(rows)
    protocol=json.loads((ROOT/'config/calibration_protocol.json').read_text())
    print('Pairs:',len(rows),'labels:',Counter(r['label'] for r in rows))
    print('Components:',len({r['component'] for r in grouped}))
    print('Base development:',json.dumps(coverage(grouped,protocol['minimums']),indent=2))
    clean=[r for r in grouped if not r['component_fitted_versions']]
    print('Strict v1/v2 source transfer:',json.dumps(coverage(clean,protocol['minimums']),indent=2))
    print('First actual question and annotation:',rows[0]['query'],rows[0]['label'])
    print(rows[0]['text'])


In [ ]:
summary=json.loads((ROOT/'eval/base_calibration_diagnostic_summary.json').read_text())
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Známá development data: žádná nezávislá acceptance ani nový práh.')


## Použitý algoritmus výběru prahu
Čteme skutečnou implementaci. Neoptimalizujeme znovu práh podle známého PCA nebo rezervovaných otázek.

In [ ]:
import inspect
from calibration_data import select_threshold
print(inspect.getsource(select_threshold))